In [1]:
import numpy as np
import os
import re

def add_coords_and_time(ds, path):
    # ------------------------
    # 1. Latitude / Longitude
    # ------------------------
    lat = np.linspace(89.875, -89.875, ds.dims["rows"])
    lon = np.linspace(-179.875, 179.875, ds.dims["columns"])

    ds = ds.assign_coords(
        lat=("rows", lat),
        lon=("columns", lon)
    ).rename({"rows": "lat", "columns": "lon"})

    # ------------------------
    # 2. Time from filename
    # ------------------------
    fname = os.path.basename(path)

    match = re.search(r"_(\d{6})_", fname)
    if match is None:
        raise ValueError(f"Cannot extract time from filename: {fname}")

    yyyymm = match.group(1)
    time = np.datetime64(f"{yyyymm[:4]}-{yyyymm[4:]}-01")

    ds = ds.assign_coords(time=time)

    return ds


In [2]:
from glob import glob
import xarray as xr

def read_netcdfs(files, dim, transform_func=None):
    def process_one_path(path):
        with xr.open_dataset(path, engine="netcdf4") as ds:
            if transform_func is not None:
                ds = transform_func(ds, path)
            ds.load()
            return ds

    paths = sorted(glob(files))
    datasets = [process_one_path(p) for p in paths]
    return xr.concat(datasets, dim)


In [3]:
ds_t = read_netcdfs(
    "/mnt/data7/nfs4/avh_ndvi/sdupuis/fire_emissions_v4_R1_1293(1)/fire_emissions_v4_R1_1293/data/GFED4.0_MQ_199*06_BA.hdf",
    dim="time",
    transform_func=add_coords_and_time
)


/tmp/ipykernel_4507/2433888565.py:15: UserWarning: rename 'rows' to 'lat' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  ).rename({"rows": "lat", "columns": "lon"})
/tmp/ipykernel_4507/2433888565.py:15: UserWarning: rename 'columns' to 'lon' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  ).rename({"rows": "lat", "columns": "lon"})
/tmp/ipykernel_4507/2433888565.py:29: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  ds = ds.assig

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

season = "JJA"

fig, axes = plt.subplots(
    nrows=1, ncols=2,
    figsize=(10, 4),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True
)

# --- Negative snow anomalies (JJA) ---
swe_neg_season_count.sel(season=season).plot(
    ax=axes[0],
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
    vmin=1,
    vmax=vmax,
    cmap=cm.lipari_r,
    levels=13
)
axes[0].set_title("Negative snow anomalies – JJA")
axes[0].coastlines()

# --- Positive snow anomalies (JJA) ---
swe_pos_season_count.sel(season=season).plot(
    ax=axes[1],
    transform=ccrs.PlateCarree(),
    add_colorbar=False,
    vmin=1,
    vmax=vmax,
    cmap=cm.lipari_r,
    levels=13
)
axes[1].set_title("Positive snow anomalies – JJA")
axes[1].coastlines()

# --- Shared colorbar ---
cbar = fig.colorbar(
    axes[0].collections[0],
    ax=axes,
    orientation="vertical",
    shrink=0.85,
    label="Count"
)

plt.savefig("swe_JJA_pos_neg.png", dpi=300)
plt.show()
